# Target Leakage & Adversarial Validation

Reference: PLAN_v2.md §3.4 — **mandatory smoke tests before final training.**

**Four checks (all blocking):**
1. **Univariate AUC scan** — flag any single feature with AUC > 0.65. Only `EXT_SOURCE_*` should legitimately exceed this.
2. **Adversarial validation** — train a classifier to distinguish train vs test. Overall AUC > 0.55 → identify drift features. Per PLAN-v2 §7 (D10): rank-transform `DAYS_*` rather than drop.
3. **Fold-encoding sanity** — out-of-fold target encodings must differ across folds for the same category.
4. **D1 neighbour sanity** — the same row's `TARGET_NEIGHBORS_500_MEAN` must differ across the 5 fold-aware indexes.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import polars as pl
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

from src import config
from src.utils import set_seed

set_seed(config.SEED)

In [ ]:
# Load the assembled main feature matrix.
main = pl.read_parquet(config.FEATURE_MATRICES['main'].parquet_path)
train_part = main.filter(pl.col(config.TARGET_COL).is_not_null())
test_part = main.filter(pl.col(config.TARGET_COL).is_null())

feature_cols = [c for c in main.columns if c not in (config.ID_COL, config.TARGET_COL)]
print(f'train rows: {train_part.height:,}, test rows: {test_part.height:,}, features: {len(feature_cols)}')

## Check 1 — Univariate AUC scan

Train one LightGBM per feature (via min/max bin trick: feed only that one column). This is slow but rigorous. For speed we instead use a single feature in a tiny LGBM with 50 rounds.

In [ ]:
y = train_part[config.TARGET_COL].to_numpy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.SEED)
tr_idx, va_idx = next(skf.split(np.zeros(len(y)), y))

univariate_auc = {}
for c in feature_cols[:200]:  # scan top 200 by default; expand to all if you have time
    x = train_part[c].to_numpy()
    if not np.issubdtype(x.dtype, np.number):
        continue
    if np.all(np.isnan(x)):
        continue
    try:
        m = lgb.LGBMClassifier(n_estimators=50, num_leaves=15, learning_rate=0.1, verbose=-1, random_state=config.SEED)
        m.fit(x[tr_idx].reshape(-1, 1), y[tr_idx])
        p = m.predict_proba(x[va_idx].reshape(-1, 1))[:, 1]
        univariate_auc[c] = roc_auc_score(y[va_idx], p)
    except Exception:
        pass

ranked = sorted(univariate_auc.items(), key=lambda kv: kv[1], reverse=True)
print('Top 20 univariate features:')
for c, a in ranked[:20]:
    flag = ' ⚠️ INSPECT' if a > 0.65 and not c.startswith('EXT_') and 'EXT_SOURCE' not in c else ''
    print(f'  {c:50s}  AUC={a:.4f}{flag}')

## Check 2 — Adversarial validation (train vs test)

Build a binary classifier where the label is `is_test`. If it scores AUC > 0.55, train and test distributions diverge.

**Action thresholds (PLAN §7 D10):**
- Adversarial AUC ≤ 0.55: ✅ no action.
- 0.55 < AUC ≤ 0.60: identify top features and rank-transform.
- AUC > 0.60: rank-transform `DAYS_*` family at minimum; investigate further.

In [ ]:
import pandas as pd

# Build adversarial dataset.
adv = pl.concat([
    train_part.select(feature_cols).with_columns(pl.lit(0).alias('is_test')),
    test_part.select(feature_cols).with_columns(pl.lit(1).alias('is_test')),
], how='vertical_relaxed')
y_adv = adv['is_test'].to_numpy()
X_adv = adv.drop('is_test').to_pandas()

skf_adv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.SEED)
oof = np.zeros(len(y_adv))
imps = pd.DataFrame({'feature': feature_cols})

for fold, (tri, vai) in enumerate(skf_adv.split(X_adv, y_adv)):
    m = lgb.LGBMClassifier(n_estimators=300, num_leaves=48, learning_rate=0.05, feature_fraction=0.5, verbose=-1, random_state=config.SEED)
    m.fit(X_adv.iloc[tri], y_adv[tri], eval_set=[(X_adv.iloc[vai], y_adv[vai])], callbacks=[lgb.early_stopping(30, verbose=False)])
    oof[vai] = m.predict_proba(X_adv.iloc[vai])[:, 1]
    imps[f'fold{fold}'] = m.booster_.feature_importance(importance_type='gain')

adv_auc = roc_auc_score(y_adv, oof)
print(f'\n=== Adversarial AUC: {adv_auc:.4f} ===')
if adv_auc > 0.6:
    print('⚠️  HIGH DRIFT — rank-transform DAYS_* features per PLAN §7 D10.')
elif adv_auc > 0.55:
    print('⚠️  Moderate drift — review top features below.')
else:
    print('✅ No significant drift.')

In [ ]:
imps['mean_gain'] = imps[[f'fold{i}' for i in range(5)]].mean(axis=1)
imps.sort_values('mean_gain', ascending=False).head(20)[['feature', 'mean_gain']]

## Check 3 — Fold-encoding sanity

If we have OOF target encodings in the matrix, verify that for the same source category, the encoded value is **different** across folds (proving fold-aware encoding worked).

In [ ]:
te_cols = [c for c in feature_cols if 'TARGET_ENC' in c.upper() or 'TE_' in c.upper()]
print(f'Target-encoded columns detected: {len(te_cols)}')
if te_cols:
    folds = pl.read_parquet(config.FEATURES_DIR / 'folds.parquet')
    # For each fold, distinct values of the encoding should differ slightly across folds for the same source category.
    print('Take this as a structural test — see src/cv.py --smoke for the assertion runner.')
else:
    print('(No TE columns yet — train.py adds them inside the CV loop)')

## Check 4 — D1 neighbour-feature sanity

For a single source row, the value of `TARGET_NEIGHBORS_500_MEAN` must differ across the 5 fold-aware index builds. If all 5 are identical → leakage.

In [ ]:
# This sanity check requires train.py to expose per-fold neighbour values to a debug parquet.
# After you run `make baseline-lgbm`, the file artifacts/predictions/d1_neighbours_per_fold.parquet
# (if produced) will hold one row per SK_ID_CURR with 5 columns showing the per-fold neighbour means.
neigh_path = config.PREDICTIONS_DIR / 'd1_neighbours_per_fold.parquet'
if neigh_path.exists():
    n = pl.read_parquet(neigh_path)
    fold_cols = [c for c in n.columns if c.startswith('fold_')]
    n_unique_per_row = n.select([pl.concat_list(fold_cols).list.n_unique().alias('uniq')])['uniq']
    pct_identical = (n_unique_per_row == 1).mean()
    print(f'Rows where all 5 fold values are identical: {pct_identical:.2%}')
    assert pct_identical < 0.01, 'D1 leakage suspected — fold-aware index not working.'
    print('✅ D1 sanity passes.')
else:
    print('(Run baseline LGBM first to populate per-fold neighbour debug parquet.)')